In [9]:
import os
import re

def extract_toc_titles(text):
    """
    Heuristically extracts possible TOC lines from the first 100 lines.
    Only picks lines that look like a chapter/article (not page numbers, not blank).
    """
    lines = text.splitlines()
    toc_lines = []
    in_toc = False
    for line in lines[:200]:
        line = line.strip()
        # Start TOC at line with 'Inhoudsopgave'
        if 'inhoud' in line.lower():
            in_toc = True
            continue
        if in_toc:
            # Stop if empty line or too short
            if not line or len(line) < 5:
                continue
            # Stop TOC if the line looks like a section break
            if re.match(r"^\s*Deel\s+\d+", line):
                continue
            # End TOC at first number (page)
            if re.match(r'^\d+$', line):
                break
            # Heuristic: line is TOC if mostly words, not numbers
            if sum(c.isalpha() for c in line) > 5:
                toc_lines.append(line)
            # If a page number at end, remove it
            if re.search(r'\d+$', line):
                toc_lines[-1] = re.sub(r'\s*\d+\s*$', '', toc_lines[-1])
        # End after 30 lines of TOC
        if in_toc and len(toc_lines) > 30:
            break
    # Clean up: remove doubles, empty, short, etc.
    toc_lines = [l.strip() for l in toc_lines if len(l.strip()) > 7]
    return toc_lines

def chapter_regex_from_titles(titles):
    """
    Build regex pattern that matches any title as a chapter boundary.
    """
    esc_titles = [re.escape(title) for title in titles]
    pattern = r'(' + '|'.join(esc_titles) + r')'
    return pattern

def split_text_on_titles(text, titles):
    """
    Split main text using TOC titles as boundaries.
    Returns list of (title, chunk) tuples.
    """
    # Build one big regex pattern
    pattern = chapter_regex_from_titles(titles)
    splits = re.split(pattern, text, flags=re.IGNORECASE)
    # Remove any preamble before first title
    if len(splits) > 1:
        splits = splits[1:]
    # Pair each chunk with the title that comes before it
    chapters = []
    for i in range(0, len(splits)-1, 2):
        title = splits[i].strip()
        chunk = splits[i+1].strip()
        chapters.append((title, chunk))
    return chapters

# Main: Process all files in folder
slaverydocument_path = "sources"
all_docs_chapters = {}

for fname in sorted(os.listdir(slaverydocument_path)):
    if fname.lower().endswith(".txt"):
        with open(os.path.join(slaverydocument_path, fname), encoding="utf-8") as f:
            text = f.read()
        titles = extract_toc_titles(text)
        chapters = split_text_on_titles(text, titles)
        all_docs_chapters[fname] = chapters
        print(f"{fname}: {len(chapters)} chapters detected.")
        print([t for t, _ in chapters])

# Example: To get all chapter texts for a file:
# chapters = all_docs_chapters['staatenslavernij.txt']  # or any file name
# for title, chunk in chapters:
#     print(f"== {title} ==\n{chunk[:300]}...\n")


Staat_slavernij.txt: 107 chapters detected.
['Het koloniale slavernijverleden en doorwerkingen: inleiding – Rose', 'Mary Allen, Esther Captain, Matthias van Rossum en Urwin Vyent', 'Actuele vraagstukken', '1. De Nederlandse wetenschap en overheid over het', 'slavernijverleden en zijn doorwerkingen – Alex van Stipriaan', '2. Een misdaad tegen de menselijkheid: Nederlandse lokale politici', 'en burgemeesters – Nancy Jouwe', 'Methode: Digital Humanities — Margo Groenewoud', '3. Het slavernijverleden in het Nederlandse onderwijs', '– Tom van der Geugten', '4. Kolonialisme en slavernij in het onderwijs: de Nederlandse', 'Cariben en Indonesië – Luc Alofs, Edu Dumasy, Kenny Meyers en', 'Elviera Sandie', 'Interviews: multiperspectiviteit in het publieke slavernijdebat', '— Myrthe Kraaijenoord en Eva Thielen', '5. Herdenkingen en doorwerkingen van de slavernij in Nederland', '– Markus Balkenhol', 'De Gouden Koets — Annemarie de Wildt', '6. Een perspectief op herstel en transformative justice – 

In [14]:
# Verzamel alle hoofdstukteksten uit alle documenten:
all_chapter_texts = []
for chapters in all_docs_chapters.values():
    for title, chunk in chapters:
        all_chapter_texts.append(chunk)
